<a href="https://colab.research.google.com/github/FrancisDoran/LG-Summarizer/blob/main/LG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build the Linked Grammar Parser from source
### Run the cell below if...
* You've just reopened the notebook
* You refreshed the notebook and get dependency errors
* If you've restarted the notebook runtime/kernel

In [1]:
#Ensure colab is using latest versions of the Ubuntu packages
!apt-get update

#Install dependencies for Linked Grammar Parser
!apt-get install -y autoconf automake libtool pkg-config flex bison swig python3-dev libpcre2-dev libedit-dev

#Pull parser source code
!git clone https://github.com/opencog/link-grammar.git

#One liner to build the parser from via the source code
!cd link-grammar && ./autogen.sh && ./configure && make -j$(nproc) && make install

zsh:1: command not found: apt-get
zsh:1: command not found: apt-get
fatal: destination path 'link-grammar' already exists and is not an empty directory.
./autogen.sh: Creating 'configure'...
libtoolize: putting auxiliary files in '.'.
libtoolize: copying file './ltmain.sh'
libtoolize: putting macros in AC_CONFIG_MACRO_DIRS, 'm4'.
libtoolize: copying file 'm4/libtool.m4'
libtoolize: copying file 'm4/ltoptions.m4'
libtoolize: copying file 'm4/ltsugar.m4'
libtoolize: copying file 'm4/ltversion.m4'
libtoolize: copying file 'm4/lt~obsolete.m4'

./autogen.sh: Running 'configure'...
checking whether configure should try to set CFLAGS/CXXFLAGS... yes
checking for a BSD-compatible install... /usr/bin/install -c
checking whether sleep supports fractional seconds... yes
checking filesystem timestamp resolution... 0.01
checking whether build environment is sane... yes
checking for a race-free mkdir -p... /usr/bin/mkdir -p
checking for gawk... gawk
checking whether make sets $(MAKE)... yes
checking

# link-gram parser demo

In [10]:
from linkgrammar import Dictionary, Sentence, ParseOptions

def desc(lkg):
  print(lkg.diagram())
  print('Postscript:')
  print(lkg.postscript())
  print('---')

dict = Dictionary('en')
po = ParseOptions(min_null_count=0, max_null_count=999)
sent = Sentence("This is a simple sentence", dict, po)
linkages = sent.parse()

for link in linkages:
  desc(link)


                   +--------Osm-------+
    +----->WV----->+  +-----Ds**x-----+
    +-->Wd---+-Ss*b+  +-PHc-+----A----+
    |        |     |  |     |         |
LEFT-WALL this.p is.v a simple.a sentence.n


Postscript:
[(LEFT-WALL)(this.p)(is.v)(a)(simple.a)(sentence.n)(RIGHT-WALL)]
[[0 6 0 (RW)][0 2 0 (WV)][0 1 0 (Wd)][1 2 0 (Ss*b)][2 5 1 (Osm)][3 5 1 (Ds**x)][3 4 2 (PHc)]
[4 5 3 (A)]]
[0]

---

                   +--------Ost-------+
    +----->WV----->+  +-----Ds**x-----+
    +-->Wd---+-Ss*b+  +-PHc-+----A----+
    |        |     |  |     |         |
LEFT-WALL this.p is.v a simple.a sentence.n


Postscript:
[(LEFT-WALL)(this.p)(is.v)(a)(simple.a)(sentence.n)(RIGHT-WALL)]
[[0 6 0 (RW)][0 2 0 (WV)][0 1 0 (Wd)][1 2 0 (Ss*b)][2 5 1 (Ost)][3 5 1 (Ds**x)][3 4 2 (PHc)]
[4 5 3 (A)]]
[0]

---

                   +--------Osm-------+
    +----->WV----->+  +-----Ds**x-----+
    +-->Wd---+-Ss*b+  |     +----A----+
    |        |     |  |     |         |
LEFT-WALL this.p is.v a simple.a sentence

In [23]:
dict = Dictionary('en')
po = ParseOptions(min_null_count=0, max_null_count=999)
sent = Sentence("This is a simple sentence", dict, po)
linkages = sent.parse()

linkages

'\n                   +--------Osm-------+\n    +----->WV----->+  +-----Ds**x-----+\n    +-->Wd---+-Ss*b+  +-PHc-+----A----+\n    |        |     |  |     |         |\nLEFT-WALL this.p is.v a simple.a sentence.n\n\n'

# Inference With a Baseline BART Model

In [34]:
import datasets
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BartForConditionalGeneration,
)
import re

#Set to CUDA gpu if available, else -> CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Constants
MODEL_ID = "facebook/bart-large-cnn"
DATASET_ID = "abisee/cnn_dailymail"
DATASET_VERSION = "3.0.0"
TEST_SPLIT = "test"

test_split = datasets.load_dataset(DATASET_ID, DATASET_VERSION, split=TEST_SPLIT)

print("\r\n")
print("Dataset format: ")
print(test_split)
print("\r\n")

print("Example Article (Sentence by Sentence): ")
article_text = test_split[1000]["article"]
sentences = re.split(r'(?<=[.!?])\s+', article_text)
for sentence in sentences:
    print(sentence.strip())
print("\r\n")

"""
TOKENIZE(R)
"""
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

"""
High-level tokenization

Handles dimensionality, and fitting.
"""

tokens = tokenizer(article_text, return_tensors="pt", max_length=512, truncation=True).to(DEVICE)
tensor = tokens["input_ids"].detach().clone().to(DEVICE)

"""
MODEL

Pretrained BART model.
"""
model = BartForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float32,
)

# ensure no gradients are calculated or applied (we are doing inference, not training)
with torch.no_grad():
  outputs = model.generate(tensor)


# use the tokenizer to decode the generated output tokens into text
for item in outputs:
    res = tokenizer.decode(item, skip_special_tokens=True)
    print("\rSummary:")
    print(res)



Dataset format: 
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 11490
})


Example Article (Sentence by Sentence): 
(CNN)Call it a little piece of heaven for a family torn apart by tragedy.
Back in July, Sierra Sharry and Lane Smith were just about to become parents.
Sharry was eight months pregnant.
But then Smith fell and hit his head.
He was taken to the OU Medical Center in Oklahoma City.
Smith never recovered.
"July 13th 2014 was the absolute worst day of my life," Sharry posted on Facebook.
"I lost my best friend.
The father of my unborn child." Their son Taos arrived a few weeks later.
When it was time for his 6-month pictures, Sharry had a special request.
Maybe the photographer could make their family complete, just for one picture .
"They asked me if I would be willing to 'play around' with capturing their first family photo by editing Taos' daddy in one of their pictures," Kayli Rene' Photography posted on Facebook.
"I just got to thinking, we don't

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summary:
Sierra Sharry and Lane Smith were about to become parents when Smith fell and hit his head. Smith never recovered. When it was time for his 6-month pictures, Sharry had a special request. "I just got to thinking, we don't have a picture with Lane in it," she said.
